# TP Integrador — Análisis de sentimientos de opiniones sobre películas

**Integrantes:** Lucas Suárez, Eric Silvestri y Walter Willich  
**Objetivo:** comparar dos modelos preentrenados usando las mismas ocho frases, inspeccionar sus resultados, medir la inferencia local y documentar el intento con `InferenceClient`.

> Ejecutar las celdas en orden. La primera carga de los modelos puede tardar y requiere conexión a internet.

## 1. Entorno e importaciones

In [ ]:
import sys
import time
import pandas as pd
import transformers
from transformers import pipeline

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("Pandas:", pd.__version__)

## 2. Carga de los dos modelos

In [ ]:
modelo_1 = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

modelo_2 = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
)

print("Los dos modelos se cargaron correctamente.")

## 3. Frases de prueba
Se usa el mismo conjunto de ocho frases para ambos modelos. Incluye ironía, jerga, neutralidad y una opinión mixta.

In [ ]:
frases = [
    "This movie was absolutely amazing!",
    "I hated every minute of this movie.",
    "La película fue excelente y me encantó.",
    "La película fue aburrida y demasiado larga.",
    "Yeah, great movie... I almost fell asleep.",
    "This movie was sick!",
    "Meh.",
    "The acting was good, but the story was terrible."
]

esperados = [
    "positive",
    "negative",
    "positive",
    "negative",
    "negative",
    "positive",
    "neutral",
    "negative"
]

print("Cantidad de frases:", len(frases))

## 4. Inferencia con `pipeline`

In [ ]:
resultados_modelo_1 = modelo_1(frases)
resultados_modelo_2 = modelo_2(frases)

print("Resultados DistilBERT:")
display(pd.DataFrame(resultados_modelo_1))

print("Resultados XLM-RoBERTa:")
display(pd.DataFrame(resultados_modelo_2))

## 5. Tabla de comparación y aciertos

In [ ]:
tabla_comparacion = pd.DataFrame({
    "Frase": frases,
    "Esperado": esperados,
    "Modelo 1": [r["label"].lower() for r in resultados_modelo_1],
    "Score M1": [r["score"] for r in resultados_modelo_1],
    "Modelo 2": [r["label"].lower() for r in resultados_modelo_2],
    "Score M2": [r["score"] for r in resultados_modelo_2],
})

tabla_comparacion["Acierto M1"] = tabla_comparacion["Modelo 1"] == tabla_comparacion["Esperado"]
tabla_comparacion["Acierto M2"] = tabla_comparacion["Modelo 2"] == tabla_comparacion["Esperado"]

display(tabla_comparacion)

aciertos_m1 = int(tabla_comparacion["Acierto M1"].sum())
aciertos_m2 = int(tabla_comparacion["Acierto M2"].sum())
print(f"DistilBERT: {aciertos_m1}/8 ({aciertos_m1/len(frases):.0%})")
print(f"XLM-RoBERTa: {aciertos_m2}/8 ({aciertos_m2/len(frases):.0%})")

## 6. Medición de tiempos con los modelos ya cargados
Estos tiempos miden una tanda de ocho frases después de cargar los modelos. No incluyen la descarga ni la inicialización inicial.

In [ ]:
inicio = time.perf_counter()
resultados_tiempo_1 = modelo_1(frases)
tiempo_total_1 = time.perf_counter() - inicio

inicio = time.perf_counter()
resultados_tiempo_2 = modelo_2(frases)
tiempo_total_2 = time.perf_counter() - inicio

mediciones = pd.DataFrame([
    {
        "Modelo": "DistilBERT",
        "Frases": len(frases),
        "Tiempo total (s)": tiempo_total_1,
        "Promedio por frase (s)": tiempo_total_1 / len(frases),
        "Condición": "Modelo ya cargado"
    },
    {
        "Modelo": "XLM-RoBERTa",
        "Frases": len(frases),
        "Tiempo total (s)": tiempo_total_2,
        "Promedio por frase (s)": tiempo_total_2 / len(frases),
        "Condición": "Modelo ya cargado"
    }
])
display(mediciones.round(4))

## 7. Tipos de retorno y acceso a los resultados

In [ ]:
resultado_individual = modelo_1("This movie was amazing!")

print("Tipo del objeto pipeline:", type(modelo_1))
print("Tipo del resultado:", type(resultado_individual))
print("Tipo del primer elemento:", type(resultado_individual[0]))
print("Resultado:", resultado_individual)
print("Etiqueta:", resultado_individual[0]["label"])
print("Puntaje:", resultado_individual[0]["score"])

resultado_individual_2 = modelo_2("La película fue excelente y me encantó.")
print("\nResultado XLM-R:", resultado_individual_2)
print("Tipo XLM-R:", type(resultado_individual_2))
print("Tipo del primer elemento XLM-R:", type(resultado_individual_2[0]))

## 8. Intento con `InferenceClient`
La API remota depende de un token configurado como variable de entorno `HF_TOKEN` y de los créditos disponibles en el proveedor. No imprimir ni guardar el valor del token.

In [ ]:
import os
from huggingface_hub import InferenceClient

token_disponible = bool(os.getenv("HF_TOKEN"))
print("HF_TOKEN configurado:", token_disponible)

if token_disponible:
    cliente = InferenceClient(token=os.getenv("HF_TOKEN"))
    try:
        resultado_api = cliente.text_classification(
            "This movie was absolutely amazing!",
            model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
        )
        print("Tipo de retorno:", type(resultado_api))
        print("Resultado remoto:", resultado_api)
    except Exception as error:
        print("La llamada remota falló.")
        print("Tipo de error:", type(error).__name__)
        print("Mensaje:", str(error))
else:
    print("No se ejecutó la llamada remota: falta configurar HF_TOKEN.")

### Interpretación del intento remoto

En la prueba realizada durante el proyecto, el proveedor devolvió **HTTP 402 Payment Required** porque no quedaban créditos de inferencia. Por eso no se obtuvo una medición remota válida y no se inventa un resultado ni un tiempo. `InferenceClient` envía una solicitud a un proveedor remoto; `pipeline` ejecuta la inferencia en el entorno local de la aplicación y necesita memoria suficiente para cargar el modelo.

## 9. Conclusiones

- Ambos modelos coincidieron con el criterio esperado en **6 de 8 frases (75 %)** en esta muestra pequeña.
- DistilBERT devuelve etiquetas `POSITIVE` o `NEGATIVE`; no dispone de una clase neutral.
- XLM-RoBERTa puede devolver `positive`, `neutral` o `negative`, pero también se equivoca con ironía y jerga.
- El score es una salida del modelo, no una garantía de que la interpretación sea correcta.
- Los tiempos deben informarse junto con las condiciones de medición. Los valores de una tanda con el modelo ya cargado no incluyen la carga inicial.
- Se eligió `pipeline` para la app porque funcionó sin depender de créditos de API remota, siempre que el servidor tenga recursos suficientes.

## 10. Uso de IA

Se utilizó ChatGPT como apoyo para diagnosticar errores, organizar el notebook y estructurar explicaciones. Los resultados deben verificarse ejecutando las celdas en el entorno de trabajo.